# XGBoost

In [12]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

In [ ]:
pip install scikit-optimize # type: ignore

In [ ]:
pip install xgboost

In [ ]:
pip install imbalanced-learn

## Metricas XGBoost

In [ ]:
# ------------------------
# Paso 1: Importar paquetes necesarios
# ------------------------
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline  # ¡Importante! usar el de imblearn
from imblearn.over_sampling import SMOTE
from joblib import dump
from time import time
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# ------------------------
# Paso 2: Cargar los datos
# ------------------------

y = Data_final['isFraud']
x = Data_final.drop(columns=['isFraud'])  # anexar base de datos de JESÚS.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=100, stratify=y)

# ------------------------
# Paso 3: Definir pipeline con SMOTE
# ------------------------
pipe_xgb_smote = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42))
])

# ------------------------
# Paso 4: Espacio de búsqueda
# ------------------------
search_spaces = {
    'xgb__n_estimators': Integer(50, 300),
    'xgb__max_depth': Integer(3, 10),
    'xgb__learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'xgb__subsample': Real(0.5, 1.0),
    'xgb__colsample_bytree': Real(0.5, 1.0),
    'xgb__gamma': Real(0, 5),
    'xgb__reg_lambda': Real(1e-3, 10, prior='log-uniform'),
    'xgb__reg_alpha': Real(1e-3, 10, prior='log-uniform')
}

# ------------------------
# Paso 5: Entrenar modelo con BayesSearchCV
# ------------------------
bayes_xgb_smote = BayesSearchCV(
    estimator=pipe_xgb_smote,
    search_spaces=search_spaces,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_xgb_smote.fit(x_train, y_train)
training_time_xgb = time() - start_time

# Guardar modelo
dump(bayes_xgb_smote, 'bayes_xgb_smote.joblib')

# ------------------------
# Paso 6: Predicciones
# ------------------------
y_pred_xgb = bayes_xgb_smote.best_estimator_.predict(x_test)
y_pred_proba_xgb = bayes_xgb_smote.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Métricas
# ------------------------
precision_xgb = precision_score(y_test, y_pred_xgb, average='weighted')
recall_xgb = recall_score(y_test, y_pred_xgb, average='weighted')
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

# ------------------------
# Paso 8: DataFrame con resultados
# ------------------------
resultados_xgb = pd.DataFrame({
    'Precision': [f"{precision_xgb:.2f}"],
    'Recall': [f"{recall_xgb:.2f}"],
    'Accuracy': [f"{accuracy_xgb:.2f}"],
    'F1-Score': [f"{f1_xgb:.2f}"],
    'AUC': [f"{auc_xgb:.2f}"],
    'CPU time (s)': [round(training_time_xgb, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo XGBoost + SMOTE (Bayesian Optimization):")
display(resultados_xgb)


In [14]:
display(resultados_xgb)

,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.95,0.94,0.94,0.95,0.83,18330.02


El modelo XGBoost muestra un rendimiento sobresaliente en la detección de fraudes. Con una precisión del 95%, el modelo es muy preciso en sus predicciones de transacciones fraudulentas. El recall del 94% indica que captura casi todas las transacciones fraudulentas reales, lo cual es crucial en un contexto de detección de fraude. La accuracy del 94% confirma su alta efectividad general en la clasificación de transacciones. El F1-Score de 0.95 representa un equilibrio excelente entre precisión y recall, demostrando una capacidad robusta para identificar fraudes. El AUC de 0.83 es particularmente prometedor, sugiriendo una muy buena capacidad para discriminar entre transacciones fraudulentas y legítimas. El tiempo de CPU de 18,330.02 segundos (aproximadamente 5.1 horas) refleja la complejidad computacional del modelo, típica de algoritmos de boosting como XGBoost, pero justificada por su alto rendimiento en la detección de fraudes.